# Detecting Structural Distress at Scale
### A Geospatial Foundation Model Approach to Urban Building Safety

**Course:** MUSA 6500 – Geospatial Machine Learning in Remote Sensing  
**Authors:** Jason Fan, Henry Sywulak-Herr

---

## Notebook Structure

| Section | Description |
|---------|-------------|
| 1 | Environment setup & imports |
| 2 | Data loading (imagery, footprints, labels, elevation) |

---
## 1. Environment Setup

In [3]:
import warnings
warnings.filterwarnings('ignore')

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import rasterio
from rasterio.plot import show as rshow

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── Add project root to path so loader scripts are importable ──────────────
PROJECT_ROOT = Path('.').resolve()   # adjust if notebook is in a sub-folder
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f'Project root: {PROJECT_ROOT}')

Device: cpu
Project root: /workspaces/musa6500-finalproject


---
## 2. Data Loading

Each dataset is loaded via its own dedicated module.  

### Parcels & Permits

In [5]:
from load_building_footprints import load_building_footprints, fetch_geojson
from pathlib import Path

VECTOR_DIR = Path('data/vector')
CRS = 'EPSG:2272'

# PWD parcels (primary geometry for all spatial joins)
parcels = load_building_footprints()
print(f'Parcels: {len(parcels):,}')

# eCLIPSE permits – used to filter false positives (new construction)
PERMIT_URL = 'https://hub.arcgis.com/api/v3/datasets/8d18914ff740444793937d8724c64da8_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1'
permits = fetch_geojson('permits', PERMIT_URL, VECTOR_DIR).to_crs(CRS)
print(f'Permits: {len(permits):,}')

[load_building_footprints] Loading parcels from cache: data/vector/parcels.geojson
[load_building_footprints] Parcels loaded: 547,290 | CRS: EPSG:2272
Parcels: 547,290
[load_building_footprints] Fetching permits from https://hub.arcgis.com/api/v3/datasets/8d18914ff740444793937d8724c64da8_0/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1 ...
  → cached to data/vector/permits.geojson
Permits: 13,899


### Training Labels (L&I Violations)

In [ ]:
from load_labels import load_labels, plot_labels

# Fetches imm_dang, unsafe, and clean_seal from Carto API,
# spatially joins to parcels, and assigns:
#   0 = Stable  |  1 = Unsafe  |  2 = Imminently Dangerous
parcels = load_labels(parcels)

plot_labels(parcels)